# 1. Preprocessing
Builds the scale-free features (set A and set B) from `Dataset_Thesis.csv` and saves them as `model_data.csv`, which all other notebooks use.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

Path("results").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

In [2]:
### load the dataset ###

data = pd.read_csv("Dataset_Thesis.csv", parse_dates=["Date"])
data = data.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(data.shape)
print(data["Date"].min().date(), "to", data["Date"].max().date())
data.groupby(["Sector", "Ticker"]).size()

(27126, 22)
2015-03-16 to 2024-12-30


Sector           Ticker
Energy           COP       2466
                 CVX       2466
                 XOM       2466
Food & Beverage  KO        2466
                 MDLZ      2466
                 PEP       2466
Technology       AAPL      2466
                 AMZN      2466
                 GOOGL     2466
                 MSFT      2466
                 TSLA      2466
dtype: int64

In [3]:
### class balance (share of upward days) ###

print("all stocks:", round(data["Target"].mean(), 4))
data.groupby("Sector")["Target"].mean().round(4)

all stocks: 0.5227


Sector
Energy             0.5076
Food & Beverage    0.5230
Technology         0.5317
Name: Target, dtype: float64

In [4]:
### feature set A: base features ###

# log returns of day t (lag 1) to day t-4 (lag 5)
for k in range(1, 6):
    data[f"ret_lag{k}"] = np.log1p(data[f"Return_lag{k}"])

data["hl_range"] = (data["High"] - data["Low"]) / data["Close"]
data["intraday_ret"] = (data["Close"] - data["Open"]) / data["Open"]

# change in log volume compared to the previous day of the same stock
previous_volume = data.groupby("Ticker")["Volume"].shift(1)
data["dlog_volume"] = np.log(data["Volume"]) - np.log(previous_volume)

In [5]:
### feature set B: technical indicators, divided by the price ###

data["dist_sma10"] = data["Close"] / data["SMA_10"] - 1
data["dist_sma50"] = data["Close"] / data["SMA_50"] - 1
data["rsi14"] = data["RSI_14"] / 100
data["macd_rel"] = data["MACD"] / data["Close"]
data["macd_signal_rel"] = data["MACD_signal"] / data["Close"]

In [6]:
### drop the first day of each stock (no previous volume) ###

FEATURES_A = ["ret_lag1", "ret_lag2", "ret_lag3", "ret_lag4", "ret_lag5",
              "hl_range", "intraday_ret", "dlog_volume"]
FEATURES_B = FEATURES_A + ["dist_sma10", "dist_sma50", "rsi14", "macd_rel", "macd_signal_rel"]

data = data.dropna(subset=FEATURES_B).reset_index(drop=True)

data["year"] = data["Date"].dt.year
data["next_ret"] = np.log(data["Next_Day_Adj_Close"] / data["Adj Close"])   # only used in the error analysis

print(data.shape)
data[FEATURES_B].describe().T.round(3)

(27115, 37)


,count,mean,std,min,25%,50%,75%,max
ret_lag1,27115.0,0.001,0.020,-0.286,-0.008,0.001,0.009,0.225
ret_lag2,27115.0,0.001,0.020,-0.286,-0.008,0.001,0.009,0.225
ret_lag3,27115.0,0.001,0.020,-0.286,-0.008,0.001,0.009,0.225
ret_lag4,27115.0,0.001,0.020,-0.286,-0.008,0.001,0.009,0.225
ret_lag5,27115.0,0.001,0.020,-0.286,-0.008,0.001,0.009,0.225
hl_range,27115.0,0.021,0.015,0.003,0.012,0.017,0.026,0.219
intraday_ret,27115.0,0.000,0.016,-0.169,-0.007,0.000,0.008,0.162
dlog_volume,27115.0,-0.000,0.325,-1.859,-0.194,-0.015,0.183,2.576
dist_sma10,27115.0,0.003,0.033,-0.362,-0.012,0.004,0.018,0.392
dist_sma50,27115.0,0.017,0.077,-0.578,-0.023,0.016,0.052,0.953


In [7]:
### save ###

columns = ["Date", "Ticker", "Sector", "year", "Target", "next_ret"] + FEATURES_B
data[columns].to_csv("model_data.csv", index=False)